# 📗 벡터 검색과 벡터 데이터베이스 구축 — ChromaDB · Qdrant

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 시간엔 코사인 유사도로 **직접 Top-K 검색기**를 만들고, 문서가 많아지면 선형 스캔이 느려 **벡터 데이터베이스**가 필요하다는 걸 배웠습니다. 이번 시간엔 실제 벡터 DB 인 **ChromaDB** 로 여행지 문서를 저장하고, **의미 기반 검색**·**메타데이터 필터**를 실습합니다. 빠른 검색의 원리인 **근사 최근접 탐색(ANN)·HNSW** 개념을 익히고, **Qdrant** 로 같은 검색을 재현해 두 도구를 비교합니다.

## ⏪ 복습 — 지난 시간: RAG 와 직접 만든 검색기

- **RAG** 는 답하기 전에 관련 문서를 **검색**해 근거로 건네는 설계(질문 → 검색기 → 생성기).
- 코사인 유사도로 **모든 문서와 비교**해 Top-K 를 뽑는 **선형 스캔** 검색기를 직접 만들었다.
- 문서가 많아지면 선형 스캔이 느려 **저장·빠른 검색·메타 필터**를 해 주는 벡터 DB 가 필요하다.
- 이번 시간에 그 벡터 DB(**ChromaDB**)를 직접 만져 본다.

**오늘의 목표**

- [ ] **의미 기반 검색**과 **키워드 검색**의 차이를 예로 설명한다.
- [ ] **ChromaDB** 가 무엇이고 어떻게 쓰는지 안다(클라이언트·컬렉션).
- [ ] 컬렉션을 만들고 문서 벡터를 **적재(add)** 한 뒤 **Top-K 검색(query)** 한다.
- [ ] **메타데이터 where 필터**로 유형·지역을 좁혀 검색한다.
- [ ] **근사 최근접 탐색(ANN)·HNSW** 가 왜 필요한지, 속도와 정확도의 맞바꿈을 이해한다.
- [ ] **Qdrant**(:memory:)로 같은 검색을 재현해 ChromaDB 와 비교한다.

> 오늘 만드는 것은 RAG 의 **검색기**입니다. 여기서 찾아온 근거를 **OpenAI API** 로 넘겨 LLM 이 답을 쓰게 하는 마지막 단계는 **교안_03** 에서 이어 붙여, 오늘 안에 **질문 → 검색 → 생성** 한 바퀴를 닫습니다.

아래 셀을 먼저 실행해 라이브러리와 한국어 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 임베딩 모델을 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

## 데이터 살펴보기 — 여행지 소개 코퍼스
지난 시간과 같은 **여행지 소개** 데이터를 씁니다. `description` 이 검색 대상 문서, `region`·`type` 은 나중에 **필터**에 쓸 메타데이터입니다.

In [ ]:
# [제공 코드] 여행지 데이터를 불러와 살펴봅니다.
spots = pd.read_csv('data/travel_spots.csv')

print("행·열 크기:", spots.shape)
print("\n[유형별 개수]"); print(spots["type"].value_counts())
display(spots.head())

# 인덱싱 타임 — 문서를 미리 임베딩해 둔다
doc_texts = spots['description'].tolist()
doc_emb = emb_model.encode(doc_texts, normalize_embeddings=True)

print("문서 임베딩 행렬:", doc_emb.shape)

## 1. 의미 기반 검색 vs 키워드 검색

<img src="images/의미검색_vs_키워드검색.png" width="900">

우리가 흔히 쓰는 검색은 **키워드 검색**입니다 — 질문에서 뽑은 **낱말이 문서에 글자 그대로 있는지**를 봅니다. 빠르고 값싸지만, 글자가 판단의 전부라 **표현이 다르면 놓치고**, 반대로 **글자만 같으면 뜻이 달라도 집어 옵니다**.

**의미 기반 검색(Semantic Search)** 은 질문과 문서를 **임베딩(벡터)** 으로 바꿔 **뜻이 가까운지**로 찾습니다. 단어가 겹치지 않아도 **의미가 비슷하면** 찾아냅니다. RAG 의 검색기가 바로 이 방식입니다.

아래 시연에서 "옛 임금이 살던 집"을 두 방식으로 검색해 비교합니다. 키워드 쪽은 질문에서 핵심 낱말 `['임금', '살던', '집']` 을 뽑아 **하나라도 들어간 문서**를 모두 가져오게 했습니다 — 즉 우리가 손으로 낱말을 골라 주고 조건도 느슨하게 잡아, **키워드 검색에 유리하게 봐준 비교**입니다.

**(가) 키워드 검색 — `str.contains`**

`str.contains` 는 넘긴 글자를 **정규표현식(regular expression)** 으로 읽습니다. 정규표현식은 '이런 모양의 글자를 찾아라'를 적는 약속된 표기법인데, 여기서는 그중 **`|`(또는)** 하나만 씁니다. `'|'.join(['임금', '살던', '집'])` 은 `'임금|살던|집'` 이라는 패턴이 되고, 이는 **"임금 또는 살던 또는 집"** 이라는 뜻입니다. 그래서 낱말 하나만 걸려도 그 문서가 뽑힙니다.

> 글자를 **그대로** 찾고 싶을 때는 `str.contains('a.b', regex=False)` 처럼 `regex=False` 를 줍니다. 그러지 않으면 `.`·`*`·`(` 같은 기호가 특별한 뜻으로 해석돼 엉뚱한 것이 걸립니다.

In [ ]:
query = "옛 임금이 살던 집"
keywords = ['임금', '살던', '집']   # 질문에서 사람이 뽑은 핵심 낱말

# '임금|살던|집' — 정규표현식에서 | 는 '또는'
pattern = '|'.join(keywords)
print("찾을 패턴:", pattern)

keyword_hits = spots[spots['description'].str.contains(pattern)]
print("[키워드 검색] 낱말", keywords, "중 하나라도 든 문서:", len(keyword_hits), "건")

# 소개 문장까지 함께 찍어야 '왜 걸렸는지' 가 눈에 보인다
for _, row in keyword_hits.iterrows():
    print(f"  {row['name']} : {row['description']} ({row['type']})")

**(나) 의미 기반 검색 — 임베딩 코사인 유사도**

같은 질문을 이번에는 **벡터로 바꿔** 비교합니다. 글자를 하나도 안 보고, 뜻이 가까운 순서로 줄을 세웁니다.

In [ ]:
# 질문도 문서와 같은 방식으로 임베딩한다
query_emb = emb_model.encode([query], normalize_embeddings=True)

# 결과가 1×N 행렬이라 [0] 으로 한 줄을 꺼낸다
sims = cosine_similarity(query_emb, doc_emb)[0]

# 부호를 뒤집어 argsort 하면 내림차순
top3 = np.argsort(-sims)[:3]

print("[의미 기반 검색] Top-3:")
for i in top3:
    print(f"  유사도 {sims[i]:.3f}  |  {spots.loc[i, 'name']} ({spots.loc[i, 'type']})")

**키워드 검색은 두 가지 방식으로 실패했습니다.**

- **놓침** — 정작 찾아야 할 **경복궁**이 한 건도 걸리지 않았습니다. 경복궁 설명은 "조선 **왕조**의 법궁으로 … **왕실**의 생활을 보여 주는 **고궁**이다" 입니다. 뜻은 정확히 '임금이 살던 집'인데, 우리가 넘긴 세 낱말('임금'·'살던'·'집')은 **글자로는 하나도 들어 있지 않습니다**.
- **오탐** — 대신 엉뚱한 문서가 **6건**이나 걸려 왔습니다. 하나같이 '집'이라는 **글자가 다른 낱말 안에 우연히 박혀 있어서** 뽑힌 것입니다.

| 걸려 온 문서 | 왜 걸렸나 |
|---|---|
| 성수동 카페거리 | 편**집**숍 |
| 인사동 거리 | 찻**집** |
| 을지로 골목 | 술**집** |
| 안동 하회마을 | 기와**집** |
| 낙안읍성 | 초가**집** |
| 감천문화마을 | 알록달록한 **집** |

임금과 아무 상관없는 카페 골목·술집 골목이 이렇게 뽑혀 올라옵니다. 키워드 검색은 **낱말의 경계를 모르기** 때문입니다.

반면 의미 기반 검색은 **경복궁을 1위**로 올리고, 그 뒤를 낙안읍성·안동 하회마을 같은 옛 건축 공간으로 채웠습니다. 낱말을 손으로 뽑아 주는 특혜까지 줬는데도 결과가 이렇습니다. 이것이 RAG 에서 벡터 검색을 쓰는 이유입니다.

### 🖐️ 함께 따라하기

이번엔 여러분이 같은 질문을 **두 방식으로 각각** 돌려 비교해 보세요. 질문은 "**등산화를 신고 오를 만한 곳**", 키워드는 `['등산화', '오를']` 입니다.

1. `hike_query` 와 `hike_keywords` 를 만든다.
2. **(가) 키워드 검색** — `spots['description'].str.contains('|'.join(hike_keywords))` 로 걸러 **건수**를 출력한다.
3. **(나) 의미 기반 검색** — 같은 질문을 임베딩해 `cosine_similarity(...)` → `np.argsort(-유사도)[:3]` 으로 Top-3 의 `name`·`type` 을 출력한다.
4. 두 결과를 눈으로 비교한다.

> 위 시연 셀처럼 **(가)와 (나)를 주석으로 확실히 나눠** 쓰세요. 두 방식이 어디서 갈라지는지가 한눈에 보여야 합니다.

In [ ]:
# 여기에 위 1~4 단계를 직접 작성해 보세요.
# (가) 키워드 검색 / (나) 의미 기반 검색 을 주석으로 나눠서 쓰면 비교가 쉽습니다.

키워드 검색은 **0건**입니다 — 뜻은 같은데 코퍼스에 쓰인 낱말이 '등산화'가 아니라 '**등산객**', '오를'이 아니라 '**오르는**'·'**등반**'이기 때문입니다. 의미 기반 검색은 같은 질문으로 지리산 노고단·설악산 울산바위·내장산 단풍길을 정확히 찾아냅니다 — **셋 다 산**입니다.

### ✅ 바로 확인 퀴즈

1. 질문의 **단어가 문서에 그대로 있는지**로 찾는 검색 방식은 무엇인가요?
2. 단어가 겹치지 않아도 **뜻이 가까우면** 찾아 주는, 임베딩 기반 검색을 무엇이라 부르나요?

<details><summary>정답 보기</summary>

1. 키워드 검색. 2. 의미 기반 검색(Semantic Search).

</details>

## 2. ChromaDB — 파이썬 친화적 로컬 벡터 DB

**ChromaDB** 는 문서 벡터를 **저장**하고 질문과 가까운 문서를 **빠르게 찾아 주는** 오픈소스 벡터 데이터베이스입니다. 지난 시간에 우리가 손으로 한 '임베딩 저장 + 코사인 Top-K'를, 더 빠르고 편하게 **대신 해 주는 도구**라고 보면 됩니다.

**이미 아는 것과의 관계**
- `doc_emb`(우리가 만든 임베딩 행렬) → ChromaDB 의 **컬렉션**에 넣어 둔다.
- `cosine_similarity` 로 하던 Top-K → 컬렉션의 `query(...)` 한 줄로 바뀐다.

**기본 사용법 (3단계)**
1. **클라이언트**를 연다: `chromadb.EphemeralClient()` — 메모리에 사는 클라이언트로, 커널을 새로 켜면 깨끗해집니다(학습·실습용).
2. **컬렉션**을 만든다: 문서 벡터를 담는 상자. `client.get_or_create_collection(이름, metadata={'hnsw:space': 'cosine'})` — 거리 기준을 **코사인**으로 지정합니다.
3. 문서를 **적재(add)** 한다: `ids`(문서 식별자), `embeddings`(우리가 만든 벡터), `documents`(원문), `metadatas`(유형·지역 등)를 함께 넣습니다.

> `create_collection(이름)` 은 **같은 이름이 이미 있으면 에러**가 납니다. 그래서 실습에선 이미 있으면 그대로 돌려주는 `get_or_create_collection` 을 써서, 셀을 여러 번 실행해도 안전하게 합니다.
> 또 우리는 **직접 만든 임베딩**(`doc_emb`)을 `embeddings=` 로 넣습니다. 이러면 ChromaDB 가 자기 기본 임베딩 모델을 쓰지 않고, 우리가 지정한 한국어 모델의 벡터를 그대로 씁니다.

<img src="images/컬렉션_네가지_데이터.png" width="980">

> 네 인자는 **각각 따로 노는 리스트가 아니라, 같은 위치끼리 한 문서를 이룹니다.** `ids[0]`·`embeddings[0]`·`documents[0]`·`metadatas[0]` 이 모두 `t01` 한 건의 정보입니다.
> 그래서 **길이나 순서가 어긋나면** 해운대의 벡터에 경포대의 설명이 붙는 식으로 조용히 망가집니다 — 네 개를 `add()` 한 번에 함께 넣는 이유입니다.

In [ ]:
# 1) 클라이언트 — 메모리에 살아서 커널을 끄면 사라진다
client = chromadb.EphemeralClient()

# 2) 컬렉션 — get_or_create 라 여러 번 실행해도 안전하다
travel_col = client.get_or_create_collection(
    'travel_guide',
    metadata={'hnsw:space': 'cosine'},   # 거리 기준
)

# 3) 적재 — 네 인자는 같은 위치끼리 한 문서를 이룬다
travel_col.add(
    ids=spots['id'].tolist(),
    embeddings=doc_emb,                        # 우리 벡터를 직접 넣는다
    documents=spots['description'].tolist(),
    metadatas=[
        {'name': n, 'region': r, 'type': t, 'entrance_fee': int(f)}
        for n, r, t, f in zip(
            spots['name'], spots['region'], spots['type'], spots['entrance_fee']
        )
    ],
)

print("컬렉션에 저장된 문서 수:", travel_col.count())

# 4) get 은 검색이 아니라 id 로 직접 꺼내는 것
peek = travel_col.get(ids=['t04'])

print('원문:', peek['documents'][0])
print('메타데이터:', peek['metadatas'][0])

### 그런데 이 데이터는 **어디에** 저장된 걸까?

방금 100건을 `add()` 했는데, 프로젝트 폴더를 열어 봐도 **새 파일이 하나도 안 생깁니다.** 우리가 연 클라이언트가 `EphemeralClient` 이기 때문입니다.

| 클라이언트 | 저장 위치 | 커널을 끄면 |
|---|---|---|
| `chromadb.EphemeralClient()` | **메모리(RAM)** — 디스크에 아무것도 안 쓴다 | **사라진다** |
| `chromadb.PersistentClient(path='./chroma_db')` | 지정한 **폴더** | 남는다(다시 열면 그대로) |
| `chromadb.HttpClient(host=..., port=...)` | **서버** 쪽 디스크 | 서버가 갖고 있다 |

실습에선 `EphemeralClient` 를 씁니다 — 커널을 새로 켤 때마다 깨끗한 상태에서 시작해야 **여러 번 실행해도 결과가 꼬이지 않기** 때문입니다.

**`PersistentClient` 를 쓰면 이런 파일들이 생깁니다.**

```
chroma_db/                       ← path= 로 준 폴더 이름
├── chroma.sqlite3               ← 문서 원문·메타데이터·컬렉션 목록 (SQLite DB 파일)
└── 0198a4f5-ab3a-.../           ← 컬렉션마다 하나씩, 이름이 UUID 인 폴더
    ├── data_level0.bin          ← 벡터 본체 + HNSW 그래프의 맨 아래층
    ├── header.bin               ← 차원·개수 같은 머리말
    ├── length.bin               ← 각 벡터의 길이 정보
    └── link_lists.bin           ← HNSW 의 '지름길 지도'(위층 연결)
```

정리하면 **원문·메타데이터는 `chroma.sqlite3` 안에**, **벡터와 HNSW 그래프는 `.bin` 파일들**에 나뉘어 들어갑니다. §5 에서 배울 HNSW 가 여기 `link_lists.bin` 으로 실제로 저장되는 것입니다. 폴더 이름이 사람이 읽을 수 없는 UUID 인 이유는, 컬렉션 이름을 바꿔도 파일은 그대로 두기 위해서입니다.

> **파일을 직접 열어 고치지 마세요.** `chroma.sqlite3` 와 `.bin` 은 서로 짝이 맞아야 합니다. 지울 때는 **폴더째** 지우고 다시 만드는 것이 안전합니다.

> 참고 — 임베딩 **모델**은 여기가 아니라 홈 폴더의 `~/.cache/huggingface/hub/` 에 따로 받아 둡니다(우리가 쓰는 한국어 모델은 400MB 남짓). 그래서 처음 한 번만 느리고, 두 번째부터는 바로 뜹니다.

### 🖐️ 함께 따라하기

위 4)에서 본 `get(ids=[...])` 을 이번엔 **다른 문서**에 써 봅시다. `t01`(해운대 해수욕장)을 꺼내 잘 들어갔는지 확인하세요.

1. `travel_col.get(ids=['t01'])` 를 호출해 결과를 `one` 에 담는다.
2. `one['documents'][0]`(원문)과 `one['metadatas'][0]`(메타데이터)을 출력해 확인한다.

In [ ]:
# 여기에 위 1~2 단계를 직접 작성해 보세요.

### ✅ 바로 확인 퀴즈

1. ChromaDB 에서 문서 벡터를 담는 상자를 무엇이라 부르나요?
2. 컬렉션에 문서를 넣을 때, ChromaDB 의 기본 임베더 대신 **우리가 만든 벡터**를 쓰려면 어떤 인자로 넣나요?

<details><summary>정답 보기</summary>

1. 컬렉션(collection). 2. `embeddings=` 인자(예: `add(..., embeddings=doc_emb)`).

</details>

## 3. Top-K 검색 — query()

이제 컬렉션에 질문을 던집니다. 질문을 임베딩해 `query_embeddings=` 로 넣고, 몇 개를 받을지 `n_results=` 로 정합니다.

```
res = travel_col.query(query_embeddings=질문벡터, n_results=3)
```

반환값 `res` 는 딕셔너리이고, 결과가 **한 겹 리스트로 감싸여** 있습니다(질문을 여러 개 넣을 수 있어서). 질문이 하나면 `[0]` 으로 꺼냅니다.
- `res['ids'][0]` — 가까운 문서 id 들
- `res['documents'][0]` — 그 원문들
- `res['distances'][0]` — **거리**(작을수록 가깝다 = 비슷하다). 코사인 거리라 0에 가까울수록 유사.
- `res['metadatas'][0]` — 그 문서들의 메타데이터

<img src="images/TopK_검색결과.png" width="960">

지난 시간의 그 질문 "바다에서 시원하게 물놀이하기 좋은 곳"을 다시 검색해, 이번엔 **ChromaDB** 가 같은 해변 문서를 찾아 주는지 봅니다.

In [ ]:
query = "바다에서 시원하게 물놀이하기 좋은 곳"
query_emb = emb_model.encode([query], normalize_embeddings=True)

res = travel_col.query(query_embeddings=query_emb, n_results=3)

print("질문:", query)

# 질문을 여러 개 넣을 수 있는 구조라 [0] 으로 한 겹 벗긴다
for doc_id, dist, meta in zip(
    res['ids'][0], res['distances'][0], res['metadatas'][0]
):
    print(f"  거리 {dist:.3f}  |  {doc_id}  {meta['name']} ({meta['type']})")

손으로 코사인 유사도를 계산했던 지난 시간과 **같은 해변 문서**가 나옵니다. 다만 이제는 `query()` **한 줄**이고, 문서가 수백만이어도 빠르게 동작합니다(그 비결은 뒤의 ANN).

### 🖐️ 함께 따라하기

질문 "**전통 음식을 파는 시장 구경**" 으로 **Top-3** 를 검색해 보세요.

1. 질문을 `food_query` 에 담아 임베딩한다.
2. `travel_col.query(query_embeddings=..., n_results=3)` 를 호출해 `food_res` 에 담는다.
3. `food_res['ids'][0]` 와 `food_res['metadatas'][0]` 를 함께 출력해 어떤 곳이 나오는지 본다.

In [ ]:
# 여기에 위 1~3 단계를 직접 작성해 보세요.

### ✅ 바로 확인 퀴즈

1. `query()` 에 질문 벡터를 넣는 인자 이름과, 받을 개수를 정하는 인자 이름은 각각 무엇인가요?
2. `res['distances'][0]` 의 값은 작을수록 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

1. `query_embeddings=` 와 `n_results=`. 2. 거리가 작을수록 질문과 **더 가깝다(더 비슷하다)**.

</details>

## 4. 메타데이터 필터 — where

벡터 검색의 진짜 힘은 **의미 검색 + 조건 필터**를 함께 거는 데 있습니다. 예를 들어 '역사 유적 중에서' 또는 '제주 지역만' 처럼 좁혀서 찾고 싶을 때, `query(..., where={...})` 로 메타데이터 조건을 겁니다.

- `where={'type': '역사'}` — 유형이 '역사'인 문서만 대상으로 Top-K.
- `where={'region': '제주'}` — 지역이 '제주'인 문서만.

<img src="images/메타필터_2단계.png" width="960">

> 필터는 **점수를 깎는 게 아니라 후보를 지웁니다.** 왼쪽처럼 전체가 후보일 때 조건을 걸면 가운데처럼 **조건에 맞는 점만 남고 나머지는 아예 사라집니다**. 그 다음에야 오른쪽처럼 질문(별)에서 가장 가까운 K개를 고릅니다.
> 그래서 **조건을 잘못 걸면 정답이 후보에서 통째로 빠져** 아무리 검색을 잘해도 못 찾습니다.

아래는 "가족과 나들이하기 좋은 곳"을 **역사 유적 안에서만** 찾는 예입니다.

In [ ]:
query = "가족과 나들이하기 좋은 곳"
query_emb = emb_model.encode([query], normalize_embeddings=True)

# where 는 점수를 깎는 게 아니라 후보에서 아예 뺀다
res = travel_col.query(query_embeddings=query_emb, n_results=3, where={'type': '역사'})

print("[type=역사 안에서] 질문:", query)
for doc_id, meta in zip(res['ids'][0], res['metadatas'][0]):
    print(f"  {doc_id}  {meta['name']}  (유형 {meta['type']}, 지역 {meta['region']})")

결과가 모두 **역사 유형**으로만 나옵니다. 의미가 가까워도 다른 유형이면 걸러진 것입니다.

### 🖐️ 함께 따라하기

이번엔 **지역**으로 필터해 봅시다. "쉬기 좋은 조용한 곳" 을 **제주 지역 안에서만** 최대 5개 검색하세요.

1. 질문을 임베딩한다.
2. `travel_col.query(query_embeddings=..., n_results=5, where={'region': '제주'})` 를 호출한다.
3. 나온 문서들의 `name` 과 `region` 을 출력해, 모두 제주인지 확인한다.

In [ ]:
# 여기에 위 1~3 단계를 직접 작성해 보세요.

제주 문서가 11건 있어서 `n_results=5` 를 꽉 채워 **5건**이 나왔고, 모두 지역이 제주입니다.

> **직접 한 번 더 해 보세요** — `where={'region': '대전'}` 으로 바꿔 같은 검색을 돌리면 `n_results=5` 를 줘도 **1건**만 나옵니다. 우리 코퍼스에 대전 문서가 하나뿐이라 필터가 후보를 하나로 줄여 버리기 때문입니다. **`n_results` 는 '최대 몇 개'이지 '반드시 몇 개'가 아닙니다** — 조건이 빡빡하면 그보다 적게 나옵니다.

### 숫자 메타데이터로 필터하기

메타데이터가 **숫자**이면 '같다'뿐 아니라 **크기 조건**도 걸 수 있습니다. 우리 여행지에는 입장료(`entrance_fee`, 원)가 숫자로 들어 있습니다(대부분 무료 0원, 고궁·유적·전망대는 유료). **유료 관광지(입장료 0원 초과)** 중에서 검색해 봅시다. 조건은 `where={'entrance_fee': {'$gt': 0}}` 처럼 **연산자**로 감쌉니다.

- `$gt`(초과)·`$gte`(이상)·`$lt`(미만)·`$lte`(이하) 로 숫자 범위를 지정합니다.

In [ ]:
query = "아이와 가 볼 만한 곳"
query_emb = emb_model.encode([query], normalize_embeddings=True)

# 숫자 조건은 {'$gt': 0} 처럼 연산자로 감싼다
res = travel_col.query(
    query_embeddings=query_emb, n_results=3, where={'entrance_fee': {'$gt': 0}}
)

print("[입장료 0원 초과 = 유료] 질문:", query)
for meta in res['metadatas'][0]:
    print(f"  {meta['name']}  (입장료 {meta['entrance_fee']}원)")

결과가 모두 **입장료가 있는** 관광지로 나옵니다. **오늘 과제(LV2)** 에서는 상품 **가격**에 이 숫자 필터를 그대로 적용해 봅니다.

### 조건을 거는 방법 — 한눈에 정리

지금까지 **문자열이 같다**(`{'type': '역사'}`)와 **숫자 비교**(`{'$gt': 0}`) 두 가지를 봤습니다. 실제로 쓸 수 있는 조건은 더 있습니다. 아래가 전부입니다.

| 쓰는 법 | 뜻 | 이 데이터에서 걸리는 건수 |
|---|---|---|
| `{'type': '역사'}` | 같다 (축약형) | 20 |
| `{'type': {'$eq': '역사'}}` | 같다 (정식) | 20 |
| `{'type': {'$ne': '역사'}}` | 같지 않다 | 80 |
| `{'entrance_fee': {'$gt': 0}}` | 초과 | 18 |
| `{'entrance_fee': {'$gte': 3000}}` | 이상 | 13 |
| `{'entrance_fee': {'$lt': 3000}}` | 미만 | 87 |
| `{'entrance_fee': {'$lte': 1000}}` | 이하 | 85 |
| `{'region': {'$in': ['제주', '부산']}}` | 목록 안에 있음 | 20 |
| `{'region': {'$nin': ['서울']}}` | 목록 밖 | 81 |
| `{'$and': [{'entrance_fee': {'$gte': 3000}}, {'entrance_fee': {'$lte': 10000}}]}` | **범위**(3000원 이상 **그리고** 10000원 이하) | 11 |
| `{'$and': [{'type': '역사'}, {'entrance_fee': {'$gt': 0}}]}` | 둘 다 만족 | 9 |
| `{'$or': [{'region': '제주'}, {'region': '부산'}]}` | 둘 중 하나 | 20 |
| `where_document={'$contains': '해변'}` | **본문에 그 글자가 들어감** | 17 |
| `where_document={'$not_contains': '해변'}` | 본문에 그 글자가 없음 | 83 |

**숫자 범위는 어떻게 거나 — 가장 헷갈리는 곳**

'3천 원 이상 1만 원 이하'처럼 **위아래를 모두 막는 조건**은 연산자가 두 개 필요합니다. 그런데 **한 딕셔너리에 두 개를 나란히 쓸 수 없습니다.**

```python
# ❌ 에러 — 연산자는 하나씩만
where={'entrance_fee': {'$gte': 3000, '$lte': 10000}}
#   ValueError: Expected operator expression to have exactly one operator

# ✅ 조건을 둘로 쪼개 $and 로 묶는다
where={'$and': [{'entrance_fee': {'$gte': 3000}},
                {'entrance_fee': {'$lte': 10000}}]}
```

**같은 열을 두 번 써도 됩니다.** `$and` 는 '조건들이 모두 참'이라는 뜻일 뿐이라, 아래쪽 경계와 위쪽 경계를 각각 하나의 조건으로 적어 묶으면 그것이 곧 범위가 됩니다. 한쪽만 막고 싶으면(예: '3천 원 이상' 만) `$and` 없이 `{'entrance_fee': {'$gte': 3000}}` 하나로 충분합니다.

**꼭 기억할 네 가지**

1. **`where` 는 메타데이터를, `where_document` 는 문서 본문을 봅니다.** 서로 **다른 인자**라 따로 쓸 수도, 함께 쓸 수도 있습니다. 그런데 `where_document` 가 하는 일은 **§1 에서 본 키워드 검색** 바로 그것입니다 — 글자가 들어 있는지만 봅니다. 즉 **의미 검색으로 순위를 매기되 특정 낱말이 든 문서만** 남기는, 두 방식을 겹쳐 쓰는 길이 열립니다. §1 에서 키워드 검색을 '실패하는 방식'으로 봤지만, **단독으로 쓰면 약하고 필터로 얹으면 쓸모 있습니다**.
2. **축약형 `{'type': '역사'}` 은 `{'type': {'$eq': '역사'}}` 의 줄임**입니다. 다만 숫자 비교는 반드시 연산자로 감싸야 합니다 — `{'entrance_fee': 3000}` 은 '3000원 **이상**'이 아니라 '**정확히 3000원**'이라는 뜻입니다(가장 흔한 실수).
3. **`$and`·`$or` 는 맨 바깥에 옵니다** — `{'$and': [조건, 조건]}` 처럼 키가 `$and` 이고 값이 조건들의 **목록**입니다. 조건 두 개를 한 딕셔너리에 나란히 쓰는 것(`{'type': '역사', 'entrance_fee': {'$gt': 0}}`)과 헷갈리기 쉬운데, ChromaDB 는 **조건이 둘 이상이면 `$and`/`$or` 로 묶기를 요구**합니다.
4. 필터는 여전히 **후보를 지우는 것**입니다. 조건이 빡빡하면 `n_results` 로 요청한 K 보다 **적게** 나올 수 있습니다.

> 아래 셀은 검색이 아니라 **건수 세기**입니다. `get(where=..., include=[])` 은 조건에 맞는 문서를 **전부** 돌려주므로(K 로 자르지 않으므로) 위 표와 맞는지 눈으로 확인할 수 있습니다.

In [ ]:
# query 가 아니라 get — 조건에 맞는 문서를 K 로 자르지 않고 전부 센다
# include=[] : id 만 받겠다는 뜻(원문·벡터를 안 받아 가볍다)

# ① $in — 목록 안에 있으면 통과
r1 = travel_col.get(where={'region': {'$in': ['제주', '부산']}}, include=[])
print('$in  제주 또는 부산      :', len(r1['ids']), '건')

# ② $and — 키가 '$and' 이고 값은 조건들의 목록
r2 = travel_col.get(
    where={'$and': [{'type': '역사'}, {'entrance_fee': {'$gt': 0}}]}, include=[]
)
print('$and 역사 + 유료         :', len(r2['ids']), '건')

# ③ 범위 — 아래 경계와 위 경계를 각각 조건으로 적어 $and 로 묶는다
r_range = travel_col.get(
    where={'$and': [
        {'entrance_fee': {'$gte': 3000}},
        {'entrance_fee': {'$lte': 10000}},
    ]},
    include=[],
)
print('범위 3000~10000원        :', len(r_range['ids']), '건')

# 한 딕셔너리에 두 연산자를 넣으면 어떻게 되는지 직접 확인
try:
    travel_col.get(where={'entrance_fee': {'$gte': 3000, '$lte': 10000}}, include=[])
except Exception as e:
    print('두 연산자를 한 번에 →', type(e).__name__, '|', str(e)[:60], '...')

# ④ where_document — 메타데이터가 아니라 문서 본문을 본다
r3 = travel_col.get(where_document={'$contains': '해변'}, include=[])
print("본문에 '해변' 이 든 문서 :", len(r3['ids']), '건')

# ⑤ 둘은 다른 인자라 함께 걸 수 있다
#    ④ 와 건수가 같은 건 우연이다 — 본문에 '해변'이 든 문서가 마침 전부 type=해변 이라 그렇다
r4 = travel_col.get(
    where={'type': '해변'}, where_document={'$contains': '해변'}, include=[]
)
print("type=해변 이면서 본문에도 '해변':", len(r4['ids']), '건')

### 🖐️ 함께 따라하기

위 표의 다른 줄도 직접 세어 표와 맞는지 확인해 보세요.

1. `$nin` — `where={'region': {'$nin': ['서울']}}` 으로 **서울이 아닌** 문서 건수를 센다.
2. `$lte` — `where={'entrance_fee': {'$lte': 1000}}` 으로 **입장료 1000원 이하** 건수를 센다.
3. `$not_contains` — `where_document={'$not_contains': '해변'}` 으로 본문에 '해변'이 **없는** 건수를 센다.

> 셋 다 `travel_col.get(..., include=[])` 로 부르고 `len(결과['ids'])` 를 출력하면 됩니다.

In [ ]:
# 여기에 위 1~3 단계를 직접 작성해 보세요.

### ✅ 바로 확인 퀴즈

1. 의미 검색에 '유형이 역사인 것만' 같은 조건을 함께 걸려면 `query()` 의 어떤 인자를 쓰나요?
2. `where={'region': '제주'}` 는 무슨 뜻인가요?
3. 입장료가 3000원 **이하**인 곳만 걸러 내려면 `where` 를 어떻게 쓰나요?
4. 메타데이터가 아니라 **문서 본문에 특정 낱말이 들어 있는지**로 거르려면 어떤 인자를 쓰나요?

<details><summary>정답 보기</summary>

1. `where=` 인자(예: `where={'type': '역사'}`). 2. 메타데이터의 `region` 이 '제주'인 문서만 검색 대상으로 삼는다. 3. `where={'entrance_fee': {'$lte': 3000}}` (`$lte` = 이하). 4. `where_document=` (예: `where_document={'$contains': '해변'}`) — §1 의 키워드 검색을 필터로 얹는 셈이다.

</details>

## 5. 빠른 검색의 비밀 — 근사 최근접 탐색(ANN)과 HNSW

지난 시간의 선형 스캔은 질문마다 **모든 문서와 비교**해 정확하지만 느립니다(문서 N개면 N번 계산). 벡터 DB 가 수백만 문서에서도 순식간에 답하는 비결은 **근사 최근접 탐색(ANN, Approximate Nearest Neighbor)** 입니다.

**핵심 아이디어**: '**정확히** 가장 가까운 것'을 고집하지 않고, '**거의** 가장 가까운 것'을 훨씬 빠르게 찾습니다. 전부 비교하는 대신, 미리 만들어 둔 **지름길 지도**를 따라 유망한 후보만 살핍니다. 그래서 **속도와 정확도를 맞바꿉니다** — 아주 조금의 정확도를 내주고 **엄청난 속도**를 얻습니다.

**HNSW**(Hierarchical Navigable Small World)는 가장 널리 쓰이는 ANN 방식입니다(ChromaDB·Qdrant 의 기본). 직관은 이렇습니다.

- 문서 벡터들을 **가까운 것끼리 연결한 그래프**로 만들어 둡니다(이웃으로 이어진 지도).
- 위층은 **성긴 지도(멀리 건너뛰는 고속도로)**, 아래층은 **촘촘한 지도(동네 골목)** 로 **여러 층**을 쌓습니다.
- 검색은 위층에서 **크게 점프**해 대략의 동네로 간 뒤, 아래층으로 내려가며 **점점 좁혀** 가까운 이웃을 찾습니다. 전부 보지 않고도 몇 번의 이동만으로 도착합니다.

<img src="images/ANN_HNSW.png" width="960">

우리가 컬렉션을 만들 때 준 `metadata={'hnsw:space': 'cosine'}` 의 `hnsw` 가 바로 이 방식이고, `cosine` 은 **거리 기준을 코사인**으로 쓴다는 뜻이었습니다. 우리가 손으로 짤 필요 없이 **ChromaDB 가 HNSW 그래프를 알아서** 만들고 검색해 줍니다.

> 정리하면 — **선형 스캔**은 느리지만 100% 정확, **ANN(HNSW)** 은 **정확도를 얼마나 내줄지 우리가 정할 수 있고** 그만큼 빨라집니다. 문서가 많은 실무에선 ANN 이 표준입니다.

그럼 이 HNSW 를 우리가 조절할 수 있는 **손잡이**는 무엇일까요. 컬렉션을 만들 때 `metadata=` 로 넘기는 값들이 그것입니다. 셋만 알면 됩니다.

### 컬렉션의 세 가지 손잡이 — 뜻과 기본값

| 손잡이 | 무엇을 정하나 | **ChromaDB 기본값** |
|---|---|---|
| `hnsw:space` | **거리를 무엇으로 재는가** — `cosine`(각도) · `l2`(직선 거리) · `ip`(내적) | **`l2`** |
| `hnsw:search_ef` | **검색할 때** 후보를 몇 개나 살펴볼지(탐색 폭) | `100` |
| `hnsw:M` | 각 지점이 가질 **이웃 연결 수**(지도의 촘촘함) | `16` |

**① `hnsw:space` — 이것만은 반드시 직접 준다**

기본값이 `cosine` 이 아니라 **`l2`** 입니다. 우리는 문장 임베딩을 **길이 1로 정규화**해 쓰므로 **각도로 재는 `cosine`** 이 맞습니다. 그래서 이 단원의 모든 컬렉션이 `metadata={'hnsw:space': 'cosine'}` 를 명시적으로 넘겼던 것입니다. 안 주면 조용히 `l2` 로 만들어져 거리 값의 의미가 달라집니다.

**② `hnsw:search_ef` — 얼마나 꼼꼼히 뒤질까**

검색할 때 후보를 몇 개나 펼쳐 보고 고를지입니다. **키우면 정확해지고 느려지고**, 줄이면 빨라지지만 진짜 가까운 문서를 놓칠 수 있습니다. 이것이 §5 첫머리에서 말한 **정확도 ↔ 속도의 맞바꿈**을 우리가 직접 돌리는 다이얼입니다.

**③ `hnsw:M` — 지도를 얼마나 촘촘히 만들까**

지점마다 이웃을 몇 개씩 이어 둘지입니다. 크게 잡으면 길이 촘촘해져 잘 찾지만 **메모리를 더 쓰고 색인이 오래 걸립니다.** 이 값은 **문서를 넣을 때 그래프를 만드는 데 쓰이므로, 나중에 바꾸려면 컬렉션을 다시 만들어야 합니다**(`search_ef` 는 검색할 때 쓰는 값이라 상대적으로 자유롭습니다).

**쓰는 법 — 컬렉션을 만들 때 `metadata=` 에 함께 넘긴다**

```python
col = client.get_or_create_collection(
    'travel_guide',
    metadata={'hnsw:space': 'cosine',   # 거리 기준 (필수로 챙길 것)
              'hnsw:search_ef': 200,   # 더 꼼꼼히 (기본 100)
              'hnsw:M': 32},           # 더 촘촘히 (기본 16)
)
```

> **실습에서는 `hnsw:space` 만 주면 충분합니다.** 나머지 둘은 기본값이 이미 무난하게 잡혀 있어서, 문서가 수십만 건으로 커져 **검색이 느리다고 느껴질 때** 비로소 건드리는 값입니다. 지금 우리 코퍼스는 100건이라 어떤 값을 줘도 결과가 같습니다.

### ✅ 바로 확인 퀴즈

1. 전부 비교하지 않고 '거의 가장 가까운' 문서를 빠르게 찾는 방식을 무엇이라 부르나요?(약자)
2. ANN 은 무엇과 무엇을 맞바꾸나요?
3. 컬렉션을 만들 때 `hnsw:space` 를 주지 않으면 거리 기준이 무엇으로 잡히나요?

<details><summary>정답 보기</summary>

1. **ANN**(근사 최근접 탐색, Approximate Nearest Neighbor). 대표 구현은 HNSW.
2. **정확도**를 내주고 **속도**를 얻는다(얼마나 내줄지는 `hnsw:search_ef` 로 조절한다).
3. **`l2`**(유클리드 거리). 우리는 각도로 재야 하므로 `'hnsw:space': 'cosine'` 을 꼭 직접 넘긴다.

</details>

## 6. 같은 검색을 Qdrant 로 재현하기

벡터 DB 는 도구가 여럿이라도 **하는 일은 같습니다**. 이번엔 오픈소스 벡터 DB **Qdrant** 로 같은 여행지 코퍼스를 색인하고 같은 질문을 검색해, ChromaDB 와 **같은 결과**가 나오는지 봅니다. Qdrant 도 `:memory:` 모드가 있어 설치 없이 메모리에서 바로 실습할 수 있습니다.

용어만 살짝 다릅니다.
- 클라이언트: `QdrantClient(':memory:')`
- 컬렉션 생성: `create_collection(이름, vectors_config=VectorParams(size=768, distance=Distance.COSINE))`
- 적재: `upsert(이름, points=[PointStruct(id=..., vector=..., payload=메타)])`
- 검색: `query_points(이름, query=질문벡터, limit=K)` → `.points` 안에 결과

> Qdrant 는 이번 시간엔 **구축·적재·검색을 체험**하는 데 목적이 있습니다(과제 채점은 ChromaDB 로 합니다).

**저장 위치도 ChromaDB 와 같은 구조로 갈립니다.**

| 쓰는 법 | 저장 위치 | ChromaDB 로 치면 |
|---|---|---|
| `QdrantClient(':memory:')` | **메모리** — 디스크에 안 쓴다 | `EphemeralClient()` |
| `QdrantClient(path='./qdrant_db')` | 지정한 **폴더** | `PersistentClient(path=...)` |
| `QdrantClient(url='http://localhost:6333')` | **서버** 쪽 디스크 | `HttpClient(...)` |

`path=` 로 폴더를 주면 이렇게 생깁니다.

```
qdrant_db/                       ← path= 로 준 폴더 이름
├── meta.json                    ← 컬렉션 목록·설정(차원·거리 기준 등)
├── .lock                        ← 두 프로세스가 동시에 열지 못하게 막는 잠금 파일
└── collection/
    └── travel_guide/            ← 컬렉션 이름이 그대로 폴더 이름이 된다
        └── storage.sqlite       ← 벡터·payload 가 함께 들어간다
```

ChromaDB 가 **원문용 `chroma.sqlite3` 와 벡터용 `.bin` 을 나눠** 두는 것과 달리, Qdrant 로컬 모드는 **`storage.sqlite` 하나에 몰아** 담습니다. 컬렉션 폴더 이름도 UUID 가 아니라 **우리가 준 이름 그대로**라 사람이 찾기 쉽습니다.

> `.lock` 때문에 **같은 폴더를 두 곳에서 동시에 열 수 없습니다.** 노트북을 두 개 켜 두면 뒤에 연 쪽이 에러가 나니, 실습 중 문제가 생기면 다른 커널을 껐는지 확인하세요. 실습에서 `:memory:` 를 쓰는 이유이기도 합니다.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# 1) 클라이언트와 컬렉션
qclient = QdrantClient(':memory:')
dim = doc_emb.shape[1]   # Qdrant 는 차원을 미리 알려 줘야 한다

# create_collection 은 이미 있으면 에러 — 지우고 다시 만든다
if qclient.collection_exists('travel_guide'):
    qclient.delete_collection('travel_guide')

qclient.create_collection(
    'travel_guide',
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

# 2) 적재 — PointStruct 하나가 문서 하나 (id·vector·payload)
qclient.upsert('travel_guide', points=[
    PointStruct(
        id=i,
        vector=doc_emb[i],
        payload={'name': spots.loc[i, 'name'], 'type': spots.loc[i, 'type']},
    )
    for i in range(len(spots))
])

print("Qdrant 에 저장된 점 개수:", qclient.count("travel_guide").count)

# 3) 검색 — ChromaDB 와 같은 질문
query = "바다에서 시원하게 물놀이하기 좋은 곳"
query_emb = emb_model.encode([query], normalize_embeddings=True)

# 질문 벡터를 리스트로 감싸지 않고 하나만 넘긴다
hits = qclient.query_points('travel_guide', query=query_emb[0], limit=3).points

print("\n[Qdrant] 질문:", query)

# score 는 Chroma 의 distance 와 반대 — 클수록 가깝다
for h in hits:
    print(f"  점수 {h.score:.3f}  |  {h.payload['name']} ({h.payload['type']})")

ChromaDB 와 **같은 해변 문서**가 **같은 순위**로 나옵니다. 숫자 표기만 다릅니다 — Chroma 는 **거리**(작을수록 가깝다), Qdrant 는 **점수**(클수록 가깝다)를 보여 줍니다.

> §3 에서 같은 질문으로 얻은 거리와 여기 점수를 **같은 문서끼리 더해 보세요 — 정확히 1 이 됩니다.** 코사인 점수 = `1 - 코사인 거리` 이기 때문입니다. 이름과 방향만 뒤집혔을 뿐 같은 값입니다.

이처럼 벡터 DB 는 **바꿔 끼워도 검색 결과는 같습니다** — 도구 선택은 '로컬이냐 서버냐', '가벼움이냐 프로덕션 성능이냐' 같은 **운영 성격**의 문제입니다.

### 🖐️ 함께 따라하기

위 데모의 payload 에는 `name`·`type` 만 넣었습니다. 이번엔 **`region`(지역)까지 넣어 다시 적재**하고, 새 질문으로 검색해 지역까지 함께 출력해 보세요.

1. 같은 컬렉션에 `upsert` 를 다시 호출한다. `payload` 에 `'region': spots.loc[i, 'region']` 을 추가한다(같은 id 로 넣으면 기존 점이 덮어써집니다).
2. 질문 "**산에 올라 단풍을 보고 싶다**" 를 임베딩한다.
3. `qclient.query_points('travel_guide', query=..., limit=3).points` 로 검색해, 각 결과의 `payload['name']`·`payload['region']` 과 `score` 를 출력한다.

In [ ]:
# 여기에 위 1~3 단계를 직접 작성해 보세요.

### ✅ 바로 확인 퀴즈

1. Qdrant 를 설치 없이 메모리에서 바로 쓰려면 클라이언트에 무엇을 넘기나요?
2. 같은 코퍼스를 ChromaDB 와 Qdrant 로 각각 검색했을 때 Top-K **순위**는 대체로 어떠했나요?

<details><summary>정답 보기</summary>

1. `':memory:'` (예: `QdrantClient(':memory:')`). 2. 거의 **같다**(도구가 달라도 임베딩·거리 기준이 같으면 검색 결과는 같다).

</details>

## 이번 강의 정리

- **의미 기반 검색**은 임베딩으로 뜻이 가까운 문서를 찾아, 단어가 겹치지 않아도 검색된다(키워드 검색과 대비).
- **ChromaDB**: `EphemeralClient` → `get_or_create_collection(…, hnsw:space='cosine')` → `add(ids, embeddings, documents, metadatas)` → `query(query_embeddings, n_results)`.
- **메타데이터 필터**는 `query(..., where={'type': '역사'})` 처럼 의미 검색에 조건을 함께 건다. 숫자 메타데이터는 `{'entrance_fee': {'$gt': 0}}` 같은 연산자(`$gt`·`$gte`·`$lt`·`$lte`)로 범위를 건다.
- **ANN(HNSW)** 은 전부 비교하지 않고 그래프 지도를 따라 '거의 가장 가까운' 문서를 빠르게 찾는다(정확도↔속도 맞바꿈).
- **Qdrant**(:memory:)로 같은 검색을 재현했다 — 도구가 달라도 결과는 같다.
- **저장 위치**는 클라이언트가 정한다. 실습의 `EphemeralClient()`·`QdrantClient(':memory:')` 는 메모리라 파일이 안 남고, `PersistentClient(path=...)`·`QdrantClient(path=...)` 는 그 폴더에 **원문·메타데이터·벡터·HNSW 그래프**를 파일로 남긴다.

## ⏭️ 예고 — 교안_03: 찾아온 근거로 답을 쓰게 하기

여기까지가 **검색기**입니다. 이어지는 **교안_03** 에서는 지금 만든 검색 결과를 **OpenAI API** 로 넘겨 LLM 이 그 근거만 보고 답을 쓰게 만듭니다. 그러면 오늘 안에 **질문 → 검색 → 생성** 한 바퀴가 닫히고, 비로소 RAG 가 완성됩니다.